In [1]:
"""
A2. Precision requirement vs κ (Classical) — full runnable notebook cell
=======================================================================
This notebook cell studies the precision (bit-truncation) required to meet a
given spectral-norm tolerance when approximating the dense Type-II NUDFT matrix.

It generates node sets using nodes_random_alternating_shift (with gamma in (0, 0.5)).
For each node set it:
  1) computes κ (a geometry parameter) from nearest-grid deviations,
  2) forms the dense F_II(t) operator,
  3) truncates each t_j to m fractional digits to obtain t_j^(m),
  4) forms F_II(t^(m)) and measures ||F_II(t) - F_II(t^(m))||_2.

For a fixed tolerance ε, it finds the minimal m such that the norm is <= ε.
It also computes the optimal low-rank expansion parameter K using find_k from
src/classical/nufft_II_lowrank, and plots m* vs log(1 + K κ) in a publication
style consistent with tests/nufft2.
"""

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
import _init_path

from pathlib import Path
import json
from typing import Dict, List, Tuple, Any

import numpy as np

from datasets.data import nodes_random_alternating_shift, nearest_grid_indices
from classical.nufft_II_lowrank import (
    find_k,
    perturbation_parameter,
    assign_closest_equispaced_gridpoint,
)


# -----------------------------------------------------------------------------
# Formatting helpers (table)
# -----------------------------------------------------------------------------
def _fmt_float(x: float) -> str:
    """Format a float for compact table printing.

    This helper converts numeric values into a short string representation
    suitable for narrow ASCII tables. It uses scientific notation for very
    small/large magnitudes and handles None and non-finite values.

    Args:
        x: Value to format.

    Returns:
        A compact string representation of the input value.

    Raises:
        None.
    """
    if x is None:
        return "-"
    if not np.isfinite(x):
        return "nan"
    ax = abs(float(x))
    if ax == 0.0:
        return "0"
    if ax < 1e-3 or ax >= 1e4:
        return f"{x:.3e}"
    return f"{x:.6g}"


def print_A2_table(rows: List[dict]) -> None:
    """Print an ASCII summary table for A2 results.

    This routine prints a compact box-style table containing per-node-set
    diagnostics and outcomes for the A2 experiment.

    Args:
        rows: List of row dictionaries with keys such as 'gamma', 'kappa',
            'K', 'log1p_Kkappa', and 'm_star'.

    Returns:
        None.

    Raises:
        None.
    """
    headers = ["#", "gamma", "kappa", "K", "log1p(Kκ)", "m*"]
    aligns = ["R", "R", "R", "R", "R", "R"]

    table_rows: List[List[str]] = []
    for i, r in enumerate(rows, 1):
        table_rows.append(
            [
                str(i),
                _fmt_float(r.get("gamma")),
                _fmt_float(r.get("kappa")),
                str(int(r.get("K"))),
                _fmt_float(r.get("log1p_Kkappa")),
                str(int(r.get("m_star"))),
            ]
        )

    # column widths
    widths = [len(h) for h in headers]
    for row in table_rows:
        for j, cell in enumerate(row):
            widths[j] = max(widths[j], len(cell))

    # box drawing
    TL, TR, BL, BR = "┌", "┐", "└", "┘"
    H, V = "─", "│"
    TJ, BJ, LJ, RJ, CJ = "┬", "┴", "├", "┤", "┼"

    def hline(left: str, mid: str, right: str) -> str:
        return left + mid.join(H * (w + 2) for w in widths) + right

    def format_cell(text: str, w: int, align: str, header: bool = False) -> str:
        if header:
            return f" {text:^{w}} "
        if align == "R":
            return f" {text:>{w}} "
        if align == "C":
            return f" {text:^{w}} "
        return f" {text:<{w}} "

    def render_row(row: List[str], header: bool = False) -> str:
        return (
            V
            + V.join(
                format_cell(row[j], widths[j], aligns[j], header=header)
                for j in range(len(headers))
            )
            + V
        )

    print(hline(TL, TJ, TR))
    print(render_row(headers, header=True))
    print(hline(LJ, CJ, RJ))
    for row in table_rows:
        print(render_row(row))
    print(hline(BL, BJ, BR))


# -----------------------------------------------------------------------------
# Precision / truncation utilities
# -----------------------------------------------------------------------------
def truncate_unit_interval_to_m_bits(t: np.ndarray, m: int) -> np.ndarray:
    """Truncate values in [0, 1) to a fixed fractional resolution.

    This routine maps each entry t_j in [0, 1) to a truncated value t_j^(m)
    defined by:
        t^(m) = floor(t * scale) / scale,
    where scale = 10^m. The result remains in [0, 1) for inputs in [0, 1).

    Args:
        t: Array of nodes in the unit interval [0, 1).
        m: Nonnegative precision parameter controlling the truncation level.

    Returns:
        An array of truncated nodes with the same shape as t.

    Raises:
        ValueError: If m is negative.
    """
    t = np.asarray(t, dtype=float)
    if m < 0:
        raise ValueError("m must be >= 0")
    scale = 10.0**m
    return np.floor(t * scale) / scale


def _wrap_to_centered_unit_interval(x: np.ndarray) -> np.ndarray:
    """Wrap values periodically into the centered unit interval.

    This routine maps inputs x to the interval [-0.5, 0.5) by periodic
    wrapping with period 1.

    Args:
        x: Input array.

    Returns:
        An array with all entries wrapped into [-0.5, 0.5).

    Raises:
        None.
    """
    return (x + 0.5) % 1.0 - 0.5


# -----------------------------------------------------------------------------
# κ computation
# -----------------------------------------------------------------------------
def compute_kappa_from_nodes(t: np.ndarray) -> Tuple[float, np.ndarray, np.ndarray]:
    """Compute the geometry parameter κ for a given node set.

    This routine computes nearest-grid indices s_j for nodes t_j, forms the
    centered circular differences, and defines:
        y*_j = 2N (t_j - s_j / N),
    where the subtraction is interpreted modulo 1 and wrapped into [-0.5, 0.5).
    It then computes:
        κ = max_j (1 - (y*_j)^2)^(-1/2).

    Args:
        t: Array of nodes in [0, 1).

    Returns:
        A tuple (kappa, y_star, s) where:
            kappa: The scalar κ value.
            y_star: The array of y*_j values.
            s: The array of nearest equispaced grid indices.

    Raises:
        None.
    """
    t = np.asarray(t, dtype=float)
    N = t.size

    s = nearest_grid_indices(t, N)
    s_over_N = s.astype(float) / N

    # shortest circular difference in [-0.5,0.5)
    diff = _wrap_to_centered_unit_interval(t - s_over_N)
    y_star = 2.0 * N * diff

    y_abs = np.abs(y_star)
    kappa = float(np.max(1.0 / np.sqrt(1.0 - y_abs * y_abs)))

    return kappa, y_star, s


# -----------------------------------------------------------------------------
# F_II(t) matrix and operator norm estimate
# -----------------------------------------------------------------------------
def FII_matrix(t: np.ndarray, normalize: bool = True) -> np.ndarray:
    """Construct the dense Type-II NUDFT matrix F_II(t).

    The matrix is defined by:
        [F_II(t)]_{k,j} = exp(-2π i k t_j),  k=0..N-1, j=0..N-1.
    If normalize is True, the matrix is scaled by 1/sqrt(N).

    Args:
        t: Array of nodes t_j in [0, 1).
        normalize: Whether to divide the matrix by sqrt(N).

    Returns:
        The dense complex NUDFT Type-II matrix of shape (N, N).

    Raises:
        None.
    """
    t = np.asarray(t, dtype=float)
    N = t.size
    k = np.arange(N, dtype=float)
    M = np.exp(-2j * np.pi * np.outer(k, t))
    if normalize:
        M /= np.sqrt(N)
    return M


def spectral_norm_power_iteration(
    A: np.ndarray,
    n_iter: int = 50,
    n_vecs: int = 3,
    seed: int = 0,
) -> float:
    """Approximate the spectral norm ||A||_2 via power iteration.

    This routine estimates the largest singular value of A by alternating
    multiplications with A and A^H, using multiple random restarts.

    Args:
        A: Input matrix.
        n_iter: Number of power iterations per restart.
        n_vecs: Number of random restarts.
        seed: Random seed for initialization.

    Returns:
        An estimate of the spectral norm ||A||_2.

    Raises:
        None.
    """
    rng = np.random.default_rng(seed)
    m, n = A.shape
    best = 0.0

    for _ in range(n_vecs):
        x = rng.standard_normal(n) + 1j * rng.standard_normal(n)
        x /= np.linalg.norm(x)

        for _ in range(n_iter):
            y = A @ x
            yn = np.linalg.norm(y)
            if yn == 0:
                break
            y /= yn

            x = A.conj().T @ y
            xn = np.linalg.norm(x)
            if xn == 0:
                break
            x /= xn

        est = float(np.linalg.norm(A @ x))
        best = max(best, est)

    return best


def matrix_diff_norm(
    t: np.ndarray,
    m: int,
    eps_norm: str = "power",
    normalize: bool = True,
) -> float:
    """Compute ||F_II(t) - F_II(t^(m))||_2 for truncated nodes.

    This routine constructs the dense Type-II NUDFT matrices for t and its
    truncated variant t^(m), forms the difference, and computes its spectral
    norm either exactly or via power iteration.

    Args:
        t: Array of nodes in [0, 1).
        m: Truncation parameter passed to truncate_unit_interval_to_m_bits.
        eps_norm: Norm evaluation mode, either 'power' or 'exact'.
        normalize: Whether to normalize F_II(t) by 1/sqrt(N).

    Returns:
        The spectral norm of the matrix difference.

    Raises:
        ValueError: If eps_norm is not 'power' or 'exact'.
    """
    F = FII_matrix(t, normalize=normalize)
    tm = truncate_unit_interval_to_m_bits(t, m)
    Fm = FII_matrix(tm, normalize=normalize)
    D = F - Fm

    if eps_norm == "exact":
        return float(np.linalg.norm(D, 2))
    if eps_norm == "power":
        return spectral_norm_power_iteration(D)
    raise ValueError("eps_norm must be 'power' or 'exact'")


def find_min_m_for_tol(
    t: np.ndarray,
    eps: float,
    m_min: int = 0,
    m_max: int = 60,
    eps_norm: str = "power",
    normalize: bool = True,
) -> Tuple[int, Dict[str, Any]]:
    """Find the minimal truncation level m satisfying a spectral-norm tolerance.

    This routine finds the smallest m such that:
        ||F_II(t) - F_II(t^(m))||_2 <= eps.
    It first linearly scans m to find a feasible upper bound, then applies a
    binary search to identify the minimal feasible m.

    Args:
        t: Array of nodes in [0, 1).
        eps: Target tolerance for the spectral norm of the matrix difference.
        m_min: Minimum m to consider.
        m_max: Maximum m to consider.
        eps_norm: Norm evaluation mode, either 'power' or 'exact'.
        normalize: Whether to normalize F_II(t) by 1/sqrt(N).

    Returns:
        A tuple (m_star, diagnostics) where:
            m_star: The minimal m satisfying the tolerance (or m_max if none).
            diagnostics: A dictionary containing 'status' and an 'errors' map.

    Raises:
        ValueError: If eps is not positive.
    """
    if eps <= 0:
        raise ValueError("eps must be > 0")

    errors: Dict[int, float] = {}

    hi = None
    for m in range(m_min, m_max + 1):
        err = matrix_diff_norm(t, m, eps_norm=eps_norm, normalize=normalize)
        errors[m] = err
        if err <= eps:
            hi = m
            break

    if hi is None:
        return m_max, {"status": "no_m_found_up_to_m_max", "errors": errors}

    lo = m_min
    while lo < hi:
        mid = (lo + hi) // 2
        if mid not in errors:
            errors[mid] = matrix_diff_norm(
                t, mid, eps_norm=eps_norm, normalize=normalize
            )
        if errors[mid] <= eps:
            hi = mid
        else:
            lo = mid + 1

    return lo, {"status": "ok", "errors": errors}


# -----------------------------------------------------------------------------
# A2 experiment runner
# -----------------------------------------------------------------------------
def run_A2_precision_vs_kappa(
    N: int = 128,
    gammas: np.ndarray | None = None,
    eps: float = 1e-8,
    eps_rank: float | None = None,
    seed: int = 0,
    m_max: int = 60,
    eps_norm: str = "power",
    normalize_FII: bool = True,
    out_dir: str | Path = "figures/A2_precision_vs_kappa",
    verbose: bool = True,
) -> Dict[str, dict]:
    """Run the A2 precision-vs-κ experiment for a fixed N.

    For each gamma in the provided list, this routine generates a node set,
    computes κ, estimates the minimal precision m* needed to satisfy the target
    tolerance eps, and computes the low-rank parameter K by:
        gamma_hat = perturbation_parameter(t, assign_closest_equispaced_gridpoint(t))
        K = find_k(gamma_hat, eps_rank).

    The results are saved to a JSON file in out_dir.

    Args:
        N: Problem size (number of nodes / transform length).
        gammas: Array of gamma values used to generate node sets. If None, a
            default linspace over [0, 0.5] is used.
        eps: Target tolerance for ||F_II(t) - F_II(t^(m))||_2.
        eps_rank: Tolerance used to compute the low-rank rank parameter K. If
            None, defaults to eps.
        seed: Random seed used for node generation.
        m_max: Maximum m to consider when searching for m*.
        eps_norm: Norm evaluation mode, either 'power' or 'exact'.
        normalize_FII: Whether to normalize F_II matrices by 1/sqrt(N).
        out_dir: Output directory for JSON results and figures.
        verbose: Whether to print status messages.

    Returns:
        A dictionary keyed by str(N), mapping to an experiment result dictionary.

    Raises:
        None.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if gammas is None:
        gammas = np.linspace(0.0, 0.5, 21)

    if eps_rank is None:
        eps_rank = eps

    rng = np.random.default_rng(seed)

    rows: List[dict] = []

    for gamma in gammas:
        t = nodes_random_alternating_shift(N=N, gamma=float(gamma), rng=rng)

        kappa, y_star, s = compute_kappa_from_nodes(t)

        s_nomod = assign_closest_equispaced_gridpoint(t)
        gamma_hat = float(perturbation_parameter(t, s_nomod))
        K = int(find_k(gamma_hat, float(eps_rank)))

        m_star, diag = find_min_m_for_tol(
            t=t,
            eps=float(eps),
            m_min=0,
            m_max=int(m_max),
            eps_norm=eps_norm,
            normalize=normalize_FII,
        )

        row = {
            "gamma": float(gamma),
            "gamma_hat": float(gamma_hat),
            "kappa": float(kappa),
            "K": int(K),
            "log1p_Kkappa": float(np.log1p(K * kappa)),
            "m_star": int(m_star),
            "status": diag["status"],
        }
        rows.append(row)

    x = np.array([r["log1p_Kkappa"] for r in rows], dtype=float)
    y = np.array([r["m_star"] for r in rows], dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    coef = np.polyfit(x[mask], y[mask], deg=1)  # [slope, intercept]

    results: Dict[str, dict] = {
        str(int(N)): {
            "N": int(N),
            "eps": float(eps),
            "eps_rank": float(eps_rank),
            "eps_norm": str(eps_norm),
            "normalize_FII": bool(normalize_FII),
            "rows": rows,
            "fit_coef": [float(coef[0]), float(coef[1])],
        }
    }

    data_path = out_dir / f"A2_precision_vs_kappa_N{N}_eps{eps}_norm-{eps_norm}.json"
    with open(data_path, "w") as f:
        json.dump(results[str(int(N))], f, indent=2)

    if verbose:
        print(f"Saved data: {data_path}")

    return results
 
# -----------------------------------------------------------------------------
# Run section (edit parameters here)
# -----------------------------------------------------------------------------
OUT_DIR = Path("figures/A2_precision_vs_kappa")

results = run_A2_precision_vs_kappa(
    N=128,
    gammas=np.linspace(0.49, 0.4999, 10),
    eps=1e-12,
    eps_rank=1e-12,  # set to eps if you want K tied to same tolerance
    seed=0,
    m_max=60,
    eps_norm="exact",  # "power" (fast) or "exact" (slow)
    normalize_FII=True,
    out_dir=OUT_DIR,
    verbose=True,
)

# Print table in nufft2 style
print_A2_table(results["128"]["rows"])

# Plot in nufft2 style
# _make_plots(results=results, out_dir=OUT_DIR, show_figures=True)

Repo root: /Users/junaida/Documents/Non-Uniform-QFT
Added to sys.path: /Users/junaida/Documents/Non-Uniform-QFT/src
Saved data: figures/A2_precision_vs_kappa/A2_precision_vs_kappa_N128_eps1e-12_norm-exact.json
┌────┬────────┬─────────┬────┬───────────┬────┐
│ #  │ gamma  │  kappa  │ K  │ log1p(Kκ) │ m* │
├────┼────────┼─────────┼────┼───────────┼────┤
│  1 │   0.49 │ 5.02519 │ 17 │   4.45931 │ 15 │
│  2 │ 0.4911 │ 5.32373 │ 17 │   4.51638 │ 13 │
│  3 │ 0.4922 │ 5.68359 │ 17 │   4.58109 │ 15 │
│  4 │ 0.4933 │ 6.12904 │ 17 │    4.6558 │ 13 │
│  5 │ 0.4944 │ 6.70032 │ 17 │   4.74411 │  7 │
│  6 │ 0.4955 │ 7.47039 │ 17 │     4.852 │ 15 │
│  7 │ 0.4966 │ 8.58954 │ 17 │   4.99058 │ 12 │
│  8 │ 0.4977 │ 10.4377 │ 17 │   5.18426 │ 15 │
│  9 │ 0.4988 │ 14.4424 │ 17 │   5.50745 │ 15 │
│ 10 │ 0.4999 │ 50.0025 │ 17 │   6.74646 │ 15 │
└────┴────────┴─────────┴────┴───────────┴────┘
